In [0]:
"""
Databricks Certified Associate Developer
Topic 4 (Deep Dive): Spark UI — Jobs, Stages, Tasks, DAG, Shuffle Read/Write

HOW SPARK EXECUTES YOUR CODE — THE FULL MENTAL MODEL
═══════════════════════════════════════════════════════════════════════════════

  Your Python code
       │
       ▼
  TRANSFORMATIONS  ← lazy, just build a recipe (DAG)
  (map, filter,
   groupBy, join)
       │
       ▼ (you call an ACTION: count, show, write, collect)
       │
       ▼
  JOB  ← one Action = one Job (visible on Spark UI "Jobs" tab)
       │
       ▼  Spark looks at the DAG and cuts it into STAGES
       │  Cut point = every wide transformation (= SHUFFLE)
       │
  ┌────┴──────────────────────────────┐
  │  STAGE 1 (Map / pre-shuffle)      │   ← "shuffle write" at end
  │  STAGE 2 (Reduce / post-shuffle)  │   ← "shuffle read" at start
  └───────────────────────────────────┘
       │
       ▼  Each stage is sliced into TASKS
       │  One TASK = one PARTITION of data processed by one CPU core
       │
  TASK 0  TASK 1  TASK 2  ...  TASK N     (parallelism = #partitions)

SPARK UI TABS YOU NEED TO KNOW:
  Jobs     → list of jobs; each row = one action call
  Stages   → stages within a job; shows shuffle read/write bytes
  Tasks    → per-partition metrics: duration, GC time, shuffle bytes
  Storage  → cached DataFrames / RDDs (size, fraction cached)
  SQL/DataFrame → visual DAG of the query plan (Spark SQL jobs only)
  Environment → all config values (spark.sql.shuffle.partitions etc.)
  Executors → executor memory, cores, GC time per executor

HOW TO ACCESS SPARK UI locally:
  http://localhost:4040   (first app; 4041, 4042 for subsequent apps)

KEY METRICS TO WATCH IN THE UI:
  Shuffle Write (MB)  → data produced by Map stage and written to disk
  Shuffle Read  (MB)  → data consumed by Reduce stage from shuffle files
  GC Time             → high GC = memory pressure, too many objects
  Task Duration       → if one task >> others → DATA SKEW
  Spill (Memory/Disk) → partition too large to fit in RAM → tune partitions
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, spark_partition_id, lit
)

spark = SparkSession.builder \
    .appName("SparkUI_DeepDive") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# ---------------------------------------------------------------------------
# SAMPLE DATA
# ---------------------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TEMP VIEW sales AS
    SELECT * FROM VALUES
        (1, 'Engineering', 'Alice',  95000),
        (2, 'Engineering', 'Carol',  110000),
        (3, 'Engineering', 'Frank',  98000),
        (4, 'Marketing',   'Bob',    72000),
        (5, 'Marketing',   'Eve',    85000),
        (6, 'HR',          'Dave',   60000),
        (7, 'HR',          'Grace',  63000),
        (8, 'Finance',     'Hank',   70000)
    AS t(id, department, name, salary)
""")
sales_df = spark.table("sales")

# ============================================================================
# SECTION 1 — WHAT IS THE DAG?
# ============================================================================
"""
DAG = Directed Acyclic Graph of transformations.

  "Directed"  → transformations flow in one direction (source → result)
  "Acyclic"   → no loops; each node depends only on its parents
  "Graph"     → nodes are RDD/DataFrame partitions; edges are transformations

Spark builds this DAG lazily — nothing runs until an ACTION is called.
The DAG is visible under:
  Spark UI → SQL/DataFrame tab → click a completed query → see the graph

RDD-level DAG is visible with: df.rdd.toDebugString()
"""
print("=" * 70)
print("SECTION 1 — RDD DAG (toDebugString)")
print("=" * 70)
print(sales_df.rdd.toDebugString().decode())
# Output shows:
#   (N) MapPartitionsRDD          ← each transformation = one node
#   (N) MapPartitionsRDD
#   (N) ExistingRDD

# ============================================================================
# SECTION 2 — NARROW vs WIDE TRANSFORMATIONS (= stage boundaries)
# ============================================================================
"""
NARROW TRANSFORMATION (no shuffle — stays in same stage):
  Each output partition depends on EXACTLY ONE input partition.
  Data does NOT move across the network.
  Examples: filter, select, withColumn, map, flatMap, coalesce

  Partition 0 ──► Partition 0   (1-to-1)
  Partition 1 ──► Partition 1
  Partition 2 ──► Partition 2

WIDE TRANSFORMATION (causes shuffle — new stage starts here):
  Each output partition depends on MULTIPLE input partitions.
  Data DOES move across the network (shuffle write → shuffle read).
  Examples: groupBy, join (SortMergeJoin), repartition, distinct, orderBy

  Partition 0 ──┐
  Partition 1 ──┼──► Partition 0 (new stage, new partition scheme)
  Partition 2 ──┘
  Partition 0 ──┐
  Partition 1 ──┼──► Partition 1
  Partition 2 ──┘

RULE: Number of stages = 1 + number of shuffles in the query
"""
print("=" * 70)
print("SECTION 2 — Narrow transformations (1 stage, 0 shuffles)")
print("=" * 70)

# All narrow — one single stage, no Exchange in plan
narrow_df = sales_df \
    .filter(col("salary") > 70000) \
    .select("department", "name", "salary") \
    .withColumn("bonus", col("salary") * lit(0.1))

narrow_df.explain()
# Plan contains: Filter → Project → Project  ← no Exchange node = no shuffle

print("=" * 70)
print("SECTION 2b — Wide transformation (groupBy → 2 stages)")
print("=" * 70)

wide_df = sales_df \
    .groupBy("department") \
    .agg(_sum("salary").alias("total_salary"))

wide_df.explain()
# Plan contains: Exchange (= shuffle) → 2 stages in Spark UI

# ============================================================================
# SECTION 3 — JOBS, STAGES, TASKS — concrete example with counts
# ============================================================================
"""
┌─────────────────────────────────────────────────────────────────────────┐
│  ACTION: wide_df.show()  → triggers JOB 1                              │
│                                                                         │
│  Stage 1 (Map/pre-shuffle):                                             │
│    Input: 1 partition (our tiny temp view)                              │
│    Work:  scan rows, compute partial sums per department                │
│    End:   SHUFFLE WRITE — writes partial results to shuffle files       │
│           (bytes visible in Spark UI → Stages tab → "Shuffle Write")   │
│                                                                         │
│  Stage 2 (Reduce/post-shuffle):                                         │
│    Start: SHUFFLE READ — reads shuffle files (possibly from network)    │
│           (bytes visible in Spark UI → Stages tab → "Shuffle Read")    │
│    Work:  merge partial sums, produce final per-department totals       │
│    Output: 4 final rows (one per department)                            │
│                                                                         │
│  Tasks per stage = number of partitions in that stage                  │
│    Stage 1 tasks = input partitions (local[*] → usually 1 per file)    │
│    Stage 2 tasks = spark.sql.shuffle.partitions (we set it to 4)       │
└─────────────────────────────────────────────────────────────────────────┘
"""
print("=" * 70)
print("SECTION 3 — Jobs, Stages, Tasks: wide_df.show() triggers 2 stages")
print("=" * 70)
wide_df.show()

# ---------------------------------------------------------------------------
# How many tasks ran in stage 2?
# spark.sql.shuffle.partitions = 4  →  4 tasks in the reduce stage
# Even if some tasks got empty partitions (data skew / low cardinality)
# ---------------------------------------------------------------------------
print("Post-shuffle partition distribution:")
wide_df \
    .withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id") \
    .agg(count("*").alias("rows")) \
    .orderBy("partition_id") \
    .show()

# ============================================================================
# SECTION 4 — SHUFFLE READ / WRITE — what they mean
# ============================================================================
"""
SHUFFLE WRITE (Map side):
  → Each executor partitions its local data by the shuffle key
    (e.g., hash(department) % numPartitions)
  → Writes partitioned data to local disk as shuffle files
  → Visible in Spark UI: Stage row → "Shuffle Write" column
  → Metric: shuffle write bytes, shuffle write records

SHUFFLE READ (Reduce side):
  → Each executor fetches its assigned partition from ALL other executors
    (network I/O — this is the expensive part)
  → Visible in Spark UI: Stage row → "Shuffle Read" column
  → Metric: shuffle read bytes, remote bytes read, fetch wait time

WHAT CAUSES HIGH SHUFFLE?
  → Many rows matching the same key (data skew → one task much slower)
  → spark.sql.shuffle.partitions too high → many tiny files, overhead
  → spark.sql.shuffle.partitions too low  → few large tasks, spill to disk

HOW TO SEE IT IN THE UI:
  Jobs tab → click Job → see Stage list with Shuffle Read/Write columns
  Stages tab → click Stage → click Task → see per-task shuffle metrics
"""
print("=" * 70)
print("SECTION 4 — Multi-stage pipeline (scan → filter → groupBy → join)")
print("=" * 70)

dept_df = spark.createDataFrame(
    [("Engineering", "NY"), ("Marketing", "Chicago"), ("HR", "Dallas"), ("Finance", "Boston")],
    "department STRING, city STRING"
)

# This pipeline creates:
#   Stage 1: scan sales_df, filter salary > 60000, partial groupBy aggregate
#   Stage 2: final groupBy aggregate (shuffle read)
#   Stage 3: broadcast join (small dept_df → no extra stage for join itself)
pipeline = sales_df \
    .filter(col("salary") > 60000) \
    .groupBy("department") \
    .agg(
        count("*").alias("headcount"),
        avg("salary").alias("avg_salary")
    ) \
    .join(dept_df, "department")

print("Plan — note Exchange operators = stage cut-points:")
pipeline.explain("formatted")
pipeline.show()

# ============================================================================
# SECTION 5 — TASKS IN DETAIL
# ============================================================================
"""
A TASK is the smallest unit of work in Spark.
  → One task processes EXACTLY ONE partition of data.
  → Sent from the Driver to an Executor core to run.
  → If you have 200 shuffle partitions → 200 tasks in that stage.

TASK LIFECYCLE:
  1. Driver serializes task and sends to executor
  2. Executor deserializes task
  3. Executor reads its partition (from memory / disk / network)
  4. Executor runs the computation
  5. Executor writes result (shuffle write or final output)
  6. Executor reports metrics back to driver

TASK STATES visible in Spark UI:
  RUNNING   → currently executing
  SUCCESS   → completed normally
  FAILED    → error; Spark retries (default 4 attempts)
  KILLED    → speculative execution killed the slower duplicate
  SKIPPED   → stage was already completed (cached result reused)

KEY TASK METRICS (Spark UI → Stages → Tasks table):
  Duration          → wall-clock time for this task
  GC Time           → time spent in Java garbage collection
  Input Size        → bytes read from disk/cache
  Shuffle Read      → bytes fetched from shuffle files (network)
  Shuffle Write     → bytes written to shuffle files
  Spill (Memory)    → data deserialized and re-serialized due to memory pressure
  Spill (Disk)      → data written to disk due to insufficient memory

TASK IMBALANCE (= DATA SKEW):
  Median task duration: 0.5 s
  Max    task duration: 45  s   ← one task 90x slower = skew problem
"""
print("=" * 70)
print("SECTION 5 — Task count = partition count per stage")
print("=" * 70)

# Each repartition sets the partition count → task count for next stage
for n_parts in [2, 4, 8]:
    df = sales_df.repartition(n_parts)
    print(f"  repartition({n_parts}) → {df.rdd.getNumPartitions()} tasks in next stage")

# ============================================================================
# SECTION 6 — STAGES THAT GET SKIPPED (caching benefit in UI)
# ============================================================================
"""
When a DataFrame is CACHED and already computed:
  → Spark UI shows stages as SKIPPED (green) instead of running them again
  → This is the visual confirmation that caching is working

Without cache:
  Job 1: Stage 1 (scan+filter) → Stage 2 (groupBy)   [runs]
  Job 2: Stage 1 (scan+filter) → Stage 2 (groupBy)   [runs again — wasteful]

With cache:
  Job 1: Stage 1 (scan+filter) → Stage 2 (groupBy)   [runs + writes to cache]
  Job 2: SKIPPED               → Stage 2 (groupBy)   [reads from cache]
"""
print("=" * 70)
print("SECTION 6 — Caching: second action skips recomputation stages")
print("=" * 70)

filtered = sales_df.filter(col("salary") > 60000).cache()

print("First action — computes + caches:")
filtered.count()

print("Second action — stages that read from cache are SKIPPED in Spark UI:")
filtered.groupBy("department").count().show()

filtered.unpersist()

# ============================================================================
# SECTION 7 — READING explain() OUTPUT AS A PROXY FOR THE SPARK UI DAG
# ============================================================================
"""
The SQL/DataFrame tab in Spark UI shows the same information as explain().
Each box in the UI = one operator node in the physical plan.

  explain() node         Spark UI DAG box
  ─────────────────────────────────────────────────
  FileScan               Scan / LocalTableScan
  Filter                 Filter
  Project                Project
  HashAggregate (partial)  Aggregate (partial)
  Exchange               Exchange (= stage boundary)
  HashAggregate (final)  Aggregate (final)
  BroadcastExchange      BroadcastExchange
  BroadcastHashJoin      BroadcastHashJoin
  SortMergeJoin          SortMergeJoin

ARROWS in UI DAG flow bottom-up (data flows from bottom = source, to top = sink).
"""
print("=" * 70)
print("SECTION 7 — explain() node map (proxy for Spark UI DAG)")
print("=" * 70)

from pyspark.sql.functions import broadcast as bc

annotated_query = sales_df \
    .filter(col("salary") > 70000) \
    .groupBy("department") \
    .agg(count("*").alias("cnt"), avg("salary").alias("avg_sal")) \
    .join(bc(dept_df), "department") \
    .orderBy(col("avg_sal").desc())

annotated_query.explain("formatted")

"""
What you'll see in the plan (bottom to top):
  LocalTableScan [id, dept, name, salary]    ← scan sales_df
  Filter (salary > 70000)                    ← narrow
  HashAggregate (partial)                    ← narrow (map-side combine)
  Exchange (hashpartitioning dept)           ← SHUFFLE = stage boundary
  HashAggregate (final)                      ← narrow (reduce-side combine)
  BroadcastExchange                          ← broadcast dept_df (no shuffle)
  BroadcastHashJoin                          ← no stage boundary!
  Sort                                       ← may add Exchange if global sort
"""
print("Final result:")
annotated_query.show()

# ============================================================================
# KEY EXAM FACTS — Spark UI
# ============================================================================
"""
EXAM QUICK REFERENCE: Spark UI
─────────────────────────────────────────────────────────────────────────
HIERARCHY:
  Application → Jobs → Stages → Tasks

  1 Action call          = 1 Job
  1 Shuffle (Exchange)   = 1 Stage boundary (stages = 1 + shuffle count)
  1 Partition processed  = 1 Task

NARROW vs WIDE:
  Narrow (no shuffle):  filter, select, withColumn, coalesce, map, flatMap
  Wide   (shuffle):     groupBy, join (SMJ), repartition, distinct, orderBy

SHUFFLE READ vs WRITE:
  Shuffle WRITE → Map stage writes partitioned data to local disk
  Shuffle READ  → Reduce stage fetches data from all executors (network I/O)
  High shuffle = costly → minimize with broadcast joins, AQE, early filters

TASK METRICS TO WATCH:
  Task duration imbalance  → skew (one task >> median)
  High GC time             → too many objects / small executor memory
  Spill (disk)             → partitions too large; increase partition count
  High fetch wait time     → network bottleneck on shuffle read

SKIPPED STAGE (green in UI):
  → Cached DataFrame was reused — stage did not re-execute

explain() → Spark UI DAG mapping:
  Exchange           → stage cut, shuffle write + read
  BroadcastExchange  → no stage cut, no shuffle on large side
  HashAggregate ×2   → 2-phase aggregation (partial in stage N, final in N+1)
─────────────────────────────────────────────────────────────────────────
"""

spark.stop()
